In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

data = [
    # replace this with the input
]
if not data: raise ValueError("No Data")
rank = data[0].replace("_", " ")
rank = rank.replace("legend", "1k") if "top" in rank else rank.replace("diamond ", "d").replace("to", "-").replace(" legend", "l").replace("1", "d1")
df = pd.DataFrame(data[1])

class_colors = {
    "Death Knight": "#6C699A",
    "DK": "#6C699A",
    "Demon Hunter": "#256F3D",
    "DH": "#256F3D",
    "Druid": "#FF7F0E",
    "Hunter": "#2CA02C",
    "Mage": "#17BECF",
    "Paladin": "#F0BD27",
    "Priest": "#C7C7C7",
    "Rogue": "#7F7F7F",
    "Shaman": "#2B7DB4",
    "lock": "#A27099",
    "Warlock": "#A27099",
    "Warrior": "#C81518"
}
df["class"] = df["name"].str.extract(f"({"|".join(class_colors.keys())})")
df = df[df["class"].notna()]
df["color"] = df["class"].map(class_colors).fillna("black")
df["class"] = df["class"].replace("lock", "Warlock").replace("Death Knight", "DK").replace("Demon Hunter", "DH")
df["is_hl"] = df["name"].str.contains("HL|Highlander")

In [ ]:
df1 = df.copy()

df1 = df1[df1["winrate"] >= 40]
df1 = df1[df1["pop"] >= 50]
#df1 = df1[df1["pop"] <= 300]
#df1 = df1[df1["is_hl"]]
#df1 = df1[df1["class"] == "Druid"]
#df1 = df1[df1["winrate"] == df1.groupby("class")["winrate"].transform("max")]

plt.figure(figsize=(25, 12))
plt.scatter(x=df1["pop"], y=df1["winrate"], c=df1["color"], s=90, edgecolor="black", alpha=0.9, zorder=3)
plt.xlabel(f"# of games ({rank} this patch)")
plt.xscale("log")
plt.ylabel("Win Rate (%)")
l1 = [m*z for e in range(len(str(min(df1["pop"])))-1,len(str(max(df1["pop"])))) for m in range(1,10) if min(df1["pop"])-(z:=10**e) < m*z < max(df1["pop"])+z]
plt.xticks(l1, l1)
r1 = range(max(0, round(min(df1["winrate"])) - 1), min(101, round(max(df1["winrate"])) + 2))
plt.yticks(r1)
if 50 in r1:
    plt.axhline(y=50, color="black", linewidth=0.6)
plt.grid()

for _, row in df1.iterrows():
    plt.text(row["pop"], row["winrate"], row["name"], fontsize=10, alpha=0.7)

plt.show()

In [ ]:
df2 = df\
.assign(total_duration=df["pop"] * df["duration"])\
.groupby("class", as_index=False)\
.agg(pop=("pop", "sum"),
    winrate=("winrate", lambda x: np.average(x, weights=df.loc[x.index, "pop"])),
    duration=("total_duration", "sum"))\
.sort_values("duration", ascending=False)

plt.figure(figsize=(9, 4))
plt.scatter(df2["pop"], df2["winrate"], c=df2["class"].map(class_colors), s=90, edgecolor="black", alpha=0.9, zorder=3)
plt.xlabel(f"# of games ({rank} this patch)")
plt.ylabel("Win Rate (%)")
r2 = range(max(0, round(min(df2["winrate"])) - 1), min(101, round(max(df2["winrate"])) + 2))
plt.yticks(r2)
if 50 in r2:
    plt.axhline(y=50, color="black", linewidth=0.6)
plt.grid()
for cls, color in class_colors.items():
    if cls not in ["DK", "DH", "lock"]:
        plt.scatter([], [], c=color, edgecolor="black", s=90, label=cls)
plt.legend(title="Classes")
plt.show()

# ------

_, (axl, axr) = plt.subplots(1, 2, figsize=(9, 4))

axr.pie(df2["duration"],
        labels=df2["class"],
        autopct=lambda p: f"{p * df2["duration"].sum() / 6000:.1f}".rstrip(".0") + "h",
        colors=df2["class"].map(class_colors),
        wedgeprops={"edgecolor": "black", "linewidth": 0.35})
axr.set_title(f"Time spent playing ({rank} this patch)")


df2["class"] = df2["class"].where(df2["pop"] / df2["pop"].sum() >= 0.04, "Other")
df2 = df2.groupby("class", as_index=False)["pop"].sum().sort_values("pop", ascending=False)

axl.pie(df2["pop"],
        labels=df2["class"],
        autopct=lambda p: f"{p / 100 * df2["pop"].sum():.0f}" + f"\n({p:.1f}".rstrip(".0") + "%)",
        colors=df2["class"].map(class_colors).fillna("pink"),
        wedgeprops={"edgecolor": "black", "linewidth": 0.35})
axl.set_title(f"Share of classes by # of games ({rank} this patch)")

plt.tight_layout()
plt.show()

In [ ]:
df3 = df[(df["winrate"] < 100) & (df["winrate"] > 0)]
#df3 = df3[df3["pop"] >= 10]

_, ax = plt.subplots(figsize=(10, 6))
order = sorted(df3["class"].unique())
sns.stripplot(data=df3, x="class", y="winrate", order=order, alpha=0)

offset_iters = {cls: iter(d) for cls, d in {cls: col.get_offsets()[:, 0] for cls, col in zip(order, ax.collections)}.items()}
x_d = [next(offset_iters[cls]) for cls in df3["class"]]

ax.scatter(x=x_d,
           y=df3["winrate"],
           s=df3["pop"] / df3["pop"].max() * 1500,
           c=df3["class"].map(class_colors),
           edgecolor="none",alpha=np.clip(1 - 0.0025 * len(df3), 0.35, 0.8))
plt.xlabel(f"Archetypes by class, scaled by popularity ({rank} this patch)")
plt.ylabel("Win Rate (%)")
plt.show()

In [ ]:
df4 = df.copy()
df4 = df4[(df4["winrate"] < 100) & (df4["winrate"] > 0)]
df4 = df[["winrate", "pop"]].melt()

with sns.axes_style("darkgrid", {"grid.color": ".64"}):
    fig = sns.FacetGrid(df4, col="variable", sharex=False, height=5)\
    .map(sns.histplot, "value", kde=True, bins=15)
    fig.set_titles("")
    plt.show()
    # TODO